# Fine-tune from YAML (KC train bundle)

This notebook runs the same training API as `python -m modeling.scripts.run_finetune`.

**Prerequisites:** Create a venv or Conda env, `pip install -r requirements-modeling.txt`, and install `jupyter` / `ipykernel`. Start Jupyter with the working directory set to the **`kc_train`** folder (the one that contains `modeling/` and `dataset/`).

The next cell adds that root to `sys.path` and `chdir`s there so imports match the CLI.

**Smoke vs full training:** Edit **`modeling/runtime_mode.py` inside the same folder as this notebook’s `ROOT`** (for this bundle that is `kc_train/modeling/runtime_mode.py`, not necessarily `thesis_prism/modeling/…` if those copies differ). Set `SMOKE_TEST = True` for fast local runs (`*_smoke.yaml`). Set `False` for full GPU/cluster runs. Or set env `MODELING_SMOKE=1`. After changing the file, **restart the kernel** or re-run the setup cell and `importlib.reload` (see code cell) so Python picks up the new value.

**If training crashes** with `Accelerator.unwrap_model() got an unexpected keyword argument 'keep_torch_compile'`, upgrade Accelerate: `pip install -U "accelerate>=1.0,<2"` (matches `requirements-modeling.txt`).

**Jupyter widget / tqdm messages** (`Could not render ... widget-view`): install `ipywidgets`, or set `MODELING_DISABLE_TQDM=1` before starting Jupyter for plain-text progress bars.

**NumPy issues:** Training metrics use **torch** for softmax/argmax, so NumPy 1.x and **2.x** both work. If you still see `ERR_IGNORE` / `umath` errors from **mixed** pip+conda installs, reinstall: `pip uninstall numpy -y && pip install numpy`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Redirect HuggingFace caches off the (small) home quota onto the lab disk.
# Must be set BEFORE any `transformers` / `datasets` import. `setdefault` keeps
# any pre-existing value from the shell, so this is also safe on a laptop
# where /data/disk1 does not exist (the env var simply won't get set there).
if "USER" in os.environ and Path("/data/disk1").is_dir():
    os.environ.setdefault("HF_HOME", "/data/disk1/{0}/hf_cache".format(os.environ["USER"]))
    os.environ.setdefault(
        "HF_DATASETS_CACHE", "/data/disk1/{0}/hf_cache/datasets".format(os.environ["USER"])
    )
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
print("HF_HOME =", os.environ.get("HF_HOME", "<unset>"))


def find_kc_train_root() -> Path:
    """Parent of `modeling/` that also has `dataset/`."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "modeling" / "trainer.py").is_file() and (candidate / "dataset").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find kc_train root. cd into the folder that contains modeling/ and dataset/, then restart."
    )


ROOT = find_kc_train_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("KC_TRAIN_ROOT =", ROOT)

# Ensure edits to modeling/*.py on disk are visible without restarting the kernel
import importlib

import modeling.runtime_mode as _runtime_mode

importlib.reload(_runtime_mode)
print("runtime_mode loaded from:", _runtime_mode.__file__)
print("SMOKE_TEST =", _runtime_mode.SMOKE_TEST, "| edit this file under ROOT/modeling/ to change it")

import numpy as _np

print("numpy", _np.__version__, "(training softmax/argmax use torch; NumPy 1.x or 2.x is fine)")

KC_TRAIN_ROOT = /Users/kingsleyeneja/Downloads/Dalhousie_MCS_courses/thesis/thesis_prism/kc_train
runtime_mode loaded from: /Users/kingsleyeneja/Downloads/Dalhousie_MCS_courses/thesis/thesis_prism/kc_train/modeling/runtime_mode.py
SMOKE_TEST = True | edit this file under ROOT/modeling/ to change it
numpy 2.2.6 (training softmax/argmax use torch; NumPy 1.x or 2.x is fine)


## Single run

Training writes to `runs/<run_name>/<language>/<timestamp>/`. Hugging Face will download pretrained weights on first use.

The config path is chosen with `resolve_train_config_path(ROOT, "xlmr_base")`: when smoke mode is on (see top), that resolves to `xlmr_base_smoke.yaml`; otherwise `xlmr_base.yaml`.

In [2]:
import traceback

from modeling.trainer import train_from_config
from modeling.runtime_mode import SMOKE_TEST, effective_smoke, resolve_train_config_path

print("SMOKE_TEST (file flag) =", SMOKE_TEST, "| effective_smoke() =", effective_smoke())
CONFIG = resolve_train_config_path(ROOT, "xlmr_base")
print("Using config:", CONFIG)
try:
    run_dir = train_from_config(config_path=CONFIG, language_override="igbo")
    print("Saved run:", run_dir)
except Exception:
    traceback.print_exc()
    raise

SMOKE_TEST (file flag) = True | effective_smoke() = True
Using config: /Users/kingsleyeneja/Downloads/Dalhousie_MCS_courses/thesis/thesis_prism/kc_train/modeling/configs/xlmr_base_smoke.yaml


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2988 [00:00<?, ? examples/s]

Map:   0%|          | 0/667 [00:00<?, ? examples/s]

/Users/kingsleyeneja/Downloads/Dalhousie_MCS_courses/thesis/thesis_prism/kc_train/modeling/trainer.py:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Phase2Trainer.__init__`. Use `processing_class` instead.
  super(WeightedLossTrainerMixin, self).__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,Roc Auc Ovr,Mcc
1,1.069600,No log,0.746627,0.465146,0.720763,0.463135,0.470409,0.745022,0.379166


Saved run: /Users/kingsleyeneja/Downloads/Dalhousie_MCS_courses/thesis/thesis_prism/kc_train/runs/xlm_roberta_base_smoke/igbo/20260507_151832


## Cross-lingual (train on one language, extra eval on the other)

Same as: `python -m modeling.scripts.run_finetune --config modeling/configs/afroxlmr_base.yaml --train-lang igbo --test-lang yoruba`

In [3]:
# from modeling.trainer import run_cross_lingual_train_eval
# from modeling.runtime_mode import resolve_train_config_path

# CONFIG = resolve_train_config_path(ROOT, "afroxlmr_base")
# run_dir = run_cross_lingual_train_eval(CONFIG, "igbo", "yoruba")
# print(run_dir)